In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import offsetbox
from datetime import date
from datetime import datetime
from dateutil.relativedelta import relativedelta
import seaborn as sns
from sklearn.preprocessing import normalize,LabelEncoder
pd.set_option('display.max_columns', 500)

### Lectura de archivos Sofia y Polizas

In [7]:
data = pd.read_csv('../Datos/sofia.csv')
ip = pd.read_csv('../Datos/ip_total.csv')
#borrar duplicados del import
ip =ip.drop(ip.columns[[28,31]], axis=1)

### Descripción de datos

 

In [8]:
print(ip.info(),'\t', data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91329 entries, 0 to 91328
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   name                       91322 non-null  object 
 1   fecha_de_nac               91329 non-null  object 
 2   antiguedad                 91329 non-null  int64  
 3   marca                      91329 non-null  object 
 4   version                    91329 non-null  object 
 5   modelo                     91329 non-null  object 
 6   tipo_de_vehiculo           91325 non-null  object 
 7   cod_infouto                91329 non-null  int64  
 8   patente                    91329 non-null  object 
 9   is_0km                     91329 non-null  int64  
 10  zip_code                   91329 non-null  int64  
 11  stated_amount              91328 non-null  float64
 12  plan_id                    91329 non-null  object 
 13  creation_date              91329 non-null  obj

In [9]:
#borrar las polizas canceladas? 
ip = ip[ip.status != 'CANCELLED']
ip = ip[ip.status != 'DESTROY']
ip = ip[ip.plan_id != 'C4']
ip = ip[ip.plan_id != 'C2']
# Polizas tipo C4 y C2 son de Cabify, no son particulares.

In [10]:
ip.status.value_counts(normalize=True)

ACTIVE      0.891222
ANNULLED    0.108778
Name: status, dtype: float64

In [11]:
# Funciones para obtener el sexo a partir del nombre.

In [12]:
# convertir series en diccionarios, lo uso para armar los diccionarios para buscar el sexo
def df_to_dict(df, key_column, val_column):
    """convierte dos pandas series en un diccionario"""
    xkey = df[key_column].tolist()
    xval = df[val_column].tolist()
    return dict(zip(xkey,xval))

In [13]:
# Lectura de datos sobre el genero de los nombres
path = '../Datos/gender.csv'
gender_list = pd.read_csv(path)  
gender_list = df_to_dict(gender_list, key_column='nombre', val_column='genero')


In [14]:
# remover denuncias duplicadas -> Las denuncias se pueden duplicar en Sofia, nos quedamos con la primera
print(data.shape)
duplicated_columns = ['company','date','license_plate','number']
mask = data.duplicated(subset = duplicated_columns, keep='first')
df = data[~mask]
print(df.shape)


(176294, 10)
(159683, 10)


### Agrupamos por placa/patente porque tenemos que volcar los datos del tipo de incidente de cada
### accidente que tuvo una patente


In [15]:
placa = df.groupby('license_plate')

In [16]:
# busco varios datos de Sofia, agrupados por patente....cuantas empresas de seguros, cuantos accidentes y tipos participo una patente.
# me quedo con el último score
score =placa.vehicle_score.last()
companies = placa.company.nunique().reset_index()
#armo la lista de tipo de accidentes 
tipos = placa.type.nunique().reset_index()
#cuento cuantas veces aparece una patente
cantidad = placa.vehicle_score.count().reset_index() # contamos cuantas veces aparece reportada una patente
pc = placa.participant_code.nunique().reset_index()

In [17]:
#renombramos la columna
cantidad =cantidad.rename(columns={'vehicle_score':'cantidad'})

#### varios joins para juntar los datos que necesito de Sofia con la tabla IP

In [18]:
placa = df.groupby('license_plate')

In [19]:
ipfull = ip.merge(companies,how='left',left_on='patente',right_on='license_plate')

In [20]:
ipfull = ipfull.merge(tipos,how='left',left_on='patente',right_on='license_plate')

In [21]:
ipfull = ipfull.merge(score,how='left',left_on='patente',right_on='license_plate')

In [22]:
ipfull =ipfull.merge(cantidad ,how='left',left_on='patente',right_on='license_plate')

In [23]:
ipfull =ipfull.merge(pc ,how='left',left_on='patente',right_on='license_plate')

In [24]:
ipfull['fecha_de_nac']= pd.to_datetime(ipfull['fecha_de_nac'])

In [25]:
ipfull['start_date']= pd.to_datetime(ipfull['start_date'],utc=True)

In [26]:
ipfull['creation_date']= pd.to_datetime(ipfull['creation_date'],utc=True)

In [27]:
ipfull = ipfull.drop(columns=['license_plate_x','license_plate_y'])

In [28]:
# Con el user creation time , hacemos un flag entre los que crearon usuario y los que no para marcarlos.


ipfull['user_created'] = np.where(ipfull.user_creation_time.isnull(),0,1)

In [29]:
now = pd.to_datetime(str(date.today()), format='%Y-%m-%d')

In [30]:
# calculamos la edad a partir de la fecha de nacimiento

ipfull['Edad'] = ipfull.apply(lambda x: relativedelta(now , x.fecha_de_nac).years,axis = 1)


In [31]:
# Funcion para obtener el sexo a partir del nombre
# si no lo puede obtener, asumimos Masculino que es mayoritario
import operator
import re 

def clean_text(txt):
    txt = re.sub("[^a-záéíóúñüäë]", " ", txt.lower())
    txt = re.sub(' +',' ', txt)
    return txt.strip().split()


def get_gender2(names):
    names = clean_text(names)
    names = [x for x in names if gender_list.get(x,'a') != 'a']
    gender ={'m':0, 'f':0, 'a':0}
    for i, name in enumerate(names):
        g = gender_list.get(name,'a')
        gender[g] += 1
        gender[g] += 2 if len(names) > 1 and i == 0 and g != 'a' else 0 
    gender['a'] = 0 if (gender['f']+gender['m']) > 0 else 1
    return max(gender.items(), key=operator.itemgetter(1))[0]

ipfull['Sexo'] = ipfull.apply(lambda x: 'm' if pd.isnull(x['name']) else get_gender2(x['name']) ,axis=1)
ipfull['Sexo'] = np.where(ipfull.Sexo == 'a','m',ipfull.Sexo)
   

In [32]:
# ver quienes tienen siniestros reportados en Iunigo y ver si son posteriores a la contratacion, o 
# cercanos a la contratacion
# 1 -los que estan en unigo - primer reporte de siniestro
en_iunigo = df.company == 'IUNIGO'
asegurados_iunigo = df[en_iunigo]
iunigo = asegurados_iunigo.groupby(['license_plate'])['date'].max() # obtiene la fecha mas lejana a la contratacion
iunigo_reports = iunigo.reset_index()

In [33]:
iunigo_reports['date']= pd.to_datetime(iunigo_reports['date'],format ="%Y-%m-%d")
ipfull =ipfull.merge(iunigo_reports ,how='left',left_on='patente',right_on='license_plate')
ipfull =ipfull.rename(columns={'date':'fecha_primer_reporte'})   ## Resguarda la fecha del primer reporte de incidente

In [34]:
# Ahora vamos a ver cuantos dias pasan desde la contratacion hasta el primer incidente.
ipfull[['fecha_primer_reporte','start_date']]
mascara = ipfull.fecha_primer_reporte.isnull()
date_time_str = '2100-01-01 00:00:00.0'
date_time_obj = datetime.strptime(date_time_str, '%Y-%m-%d %H:%M:%S.%f')

In [35]:
# poner fechas donde no hubo incidente... mandarlas al futuro para separarlos de los que tuvieron incidentes
# eliminar horas, minutos y segundos
ipfull.fecha_primer_reporte.loc[mascara] = date_time_obj
ipfull.fecha_primer_reporte = ipfull.fecha_primer_reporte.apply(lambda x: x.replace(hour=0,minute=0,second = 0))
ipfull.start_date = ipfull.start_date.apply(lambda x: x.replace(hour=0,minute=0,second = 0))
ipfull.fecha_primer_reporte =pd.to_datetime(ipfull.fecha_primer_reporte,utc=True)

/Users/maruiz/opt/anaconda3/envs/xgboost/lib/python3.7/site-packages/pandas/core/indexing.py:671: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


In [36]:
# 2 ver si la fecha del primer incidente reportado esta dentro de los primeros 31 dias.. Si es el caso, La anulacion viene por Iunigo

ipfull['dias_para_reporte_inicial'] =  (ipfull.fecha_primer_reporte - ipfull.start_date)
ipfull.dias_para_reporte_inicial = ipfull.dias_para_reporte_inicial.apply(lambda x : x.days)
# hay gente que ha tenido polizas con iunigo con incidentes en el pasado, y son polizas anteriores,
# esto da negativo en la cuenta, por eso hay que llevarlas  a futuro.
ipfull['ventana_31_dias'] = ipfull.apply(lambda x: 0 if x.dias_para_reporte_inicial > 31 else 1, axis = 1)
mascara = ipfull.ventana_31_dias < 0
ipfull.ventana_31_dias.loc[mascara] = 9999  # los que no tuvieron les ponemos 9999

In [37]:
# tratamiento de la antiguedad del vehiculo
ipfull.antiguedad = ipfull.antiguedad.astype(int)

In [38]:
ipfull['antiguedad_años'] =  now.year -ipfull.antiguedad 

In [39]:
#  crear la columna target, que seria 0- normal , 1 - anulada, luego vemos los casos especiales de anulacion
ipfull['target'] = ipfull['status'].apply(lambda x: 0 if x != 'ANNULLED' else 1)

In [40]:
# Me interesa Auto contra el resto.
# por eso, lo que no se a auto es "otros"
ipfull['tipo_de_vehiculo'] = np.where(ipfull.tipo_de_vehiculo =='AUTO','auto','otro')


In [41]:
#planId  son en secuencia p4,p1,p2,p3
#Rol a,b,c,d,e,f,g invertido  --> es la categoria del BCRA para el tomador de la poliza
ipfull.rol.value_counts()
rol_dict = {"A":9,"B":8,"C":7,"D":6,"E":5,"F":4,"G":3,"H":2,"I":1,"N":0}
plan_dict = {"P4":2,"P1":3,"P2":4,"P3":5}  # Tipo de producto contratado

In [42]:
# Rol es el riesgo del banco central
ipfull.rol.replace(rol_dict, inplace=True)
ipfull.plan_id.replace(plan_dict, inplace=True)

In [43]:
ipfull.rename(columns={'rol':'riesgo_bcra'},inplace= True)
ipfull.rename(columns={'type':'tipo'},inplace= True)

In [44]:
# Eliminamos outliers del monto asegurado
print(ipfull.shape)
data_mean, data_std = ipfull.stated_amount.mean(), ipfull.stated_amount.std()
# identificar outliers
cut_off = data_std * 3  # al 95%
lower, upper = data_mean - cut_off, data_mean + cut_off
no_outliers  = np.logical_and( ipfull.stated_amount >= lower ,ipfull.stated_amount <= upper)
ipfull = ipfull[no_outliers]
print(ipfull.shape)

(69711, 51)
(68380, 51)


In [45]:
# si no tuvo accidentes, relleno de ceros la cantidad.  no puedo hacer el proceso de outliers.

ipfull.cantidad = ipfull.cantidad.fillna(0)


In [46]:
# Eliminamos outliers de cantidad de accidentes
data_mean, data_std = ipfull.cantidad.mean(), ipfull.cantidad.std()
# identificar outliers
cut_off = data_std * 3  # al 99.5%
lower, upper = data_mean - cut_off, data_mean + cut_off

In [47]:
no_outliers  = (ipfull.cantidad <= upper)

In [48]:
ipfull = ipfull[no_outliers]


In [53]:
# Guardo el dataset integrado
ipfull = ipfull.rename(columns={'company':'cantidad_aseguradoras_previas',
                        'type':'tipo_siniestro',
                        'cantidad':'cantidad_siniestros'})
# blanqueamos el nomre y apellido
ipfull.mame = 'XXXXX'
ipfull.to_csv('../Datos/ipfull.csv',index=False)
ipfull.to_csv('../Datos/ip_total.csv',index=False)

In [50]:
# y Selecciono las columnas que a-priori quiero usar en el modelo

In [51]:
columnas = [
       'is_0km',
       'stated_amount',
       'plan_id',
       'zip_code',
       'riesgo_bcra', 
       'cantidad_aseguradoras_previas', 
       'vehicle_score',
       'cantidad_siniestros',
       'tipo',
       'participant_code',
       'Edad',  
       'Sexo',  
       'antiguedad_años',
       'ventana_31_dias',
       'tipo_de_vehiculo',
       'platform',
       'user_created',
       'target']
to_process = ipfull[columnas]
to_process.to_csv('../Datos/ip_a_procesar.csv',index=False)